# CIFAR-100 + Places — All Methods (bs=1, matching paper)

Runs all method variants on the hardest pair with paper-matching settings.

In [2]:
#@title **1. Setup (skip if already done in another notebook)**
import os, sys, shutil, subprocess, re, time

REPO = '/content/ZS-NTTA'
DATA = '/content/datasets'

if not os.path.exists(REPO):
    !git clone https://github.com/tmlr-group/ZS-NTTA.git {REPO}
else:
    print('Repo exists.')

os.chdir(REPO)
!pip install -q ml-collections absl-py ftfy wandb seaborn scikit-learn regex

import torch
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE"}')

Repo exists.
GPU: Tesla T4


In [3]:
#@title **2. Patch paths (skip if already done)**
os.makedirs(DATA, exist_ok=True)

cfg_path = os.path.join(REPO, 'configs/default_configs.py')
with open(cfg_path, 'r') as f:
    txt = f.read()
OLD = '/data2/chentao/data/check_data'
if OLD in txt:
    txt = txt.replace(OLD, DATA)
    with open(cfg_path, 'w') as f:
        f.write(txt)
    print('Patched default_configs.py')
else:
    print('Already patched.')

DCN = os.path.join(REPO, 'data/dataset_class_names.py')
DCN_BAK = os.path.join(REPO, 'data/dataset_class_names_ORIGINAL.py')
if not os.path.exists(DCN_BAK):
    shutil.copy2(DCN, DCN_BAK)

dcn_code = f'''import os\nimport warnings\nfrom .imagenet_prompts import imagenet_classes, cifar10_classes, cifar100_classes\nfrom .fewshot_datasets import fewshot_datasets\nfrom .imagenet_variants import thousand_k_to_200, imagenet_a_mask, imagenet_r_mask, imagenet_v_mask\n\nroot = "{DATA}"\n\nbird200_classes = []\ntry:\n    import pandas as pd\n    _p = os.path.join(root, "CUB_200_2011", "classes.txt")\n    if os.path.exists(_p):\n        _cn = pd.read_csv(_p, sep=" ", names=["class_id", "target"])\n        bird200_classes = [n.split(".")[1].replace("_", " ") for n in _cn.target]\n    else:\n        warnings.warn(f"bird200 not found at {{_p}}")\nexcept Exception as e:\n    warnings.warn(f"bird200: {{e}}")\n\ncar196_classes = []\ntry:\n    import scipy.io as sio\n    _p = os.path.join(root, "stanford_cars/devkit", "cars_meta.mat")\n    if os.path.exists(_p):\n        car196_classes = sio.loadmat(_p, squeeze_me=True)["class_names"].tolist()\n    else:\n        warnings.warn(f"car196 not found at {{_p}}")\nexcept Exception as e:\n    warnings.warn(f"car196: {{e}}")\n\nfood101_classes = ["Apple pie", "Baby back ribs", "Baklava", "Beef carpaccio", "Beef tartare", "Beet salad", "Beignets", "Bibimbap", "Bread pudding", "Breakfast burrito", "Bruschetta", "Caesar salad", "Cannoli", "Caprese salad", "Carrot cake", "Ceviche", "Cheesecake", "Cheese plate", "Chicken curry", "Chicken quesadilla", "Chicken wings", "Chocolate cake", "Chocolate mousse", "Churros", "Clam chowder", "Club sandwich", "Crab cakes", "Creme brulee", "Croque madame", "Cup cakes", "Deviled eggs", "Donuts", "Dumplings", "Edamame", "Eggs benedict", "Escargots", "Falafel", "Filet mignon", "Fish and chips", "Foie gras", "French fries", "French onion soup", "French toast", "Fried calamari", "Fried rice", "Frozen yogurt", "Garlic bread", "Gnocchi", "Greek salad", "Grilled cheese sandwich", "Grilled salmon", "Guacamole", "Gyoza", "Hamburger", "Hot and sour soup", "Hot dog", "Huevos rancheros", "Hummus", "Ice cream", "Lasagna", "Lobster bisque", "Lobster roll sandwich", "Macaroni and cheese", "Macarons", "Miso soup", "Mussels", "Nachos", "Omelette", "Onion rings", "Oysters", "Pad thai", "Paella", "Pancakes", "Panna cotta", "Peking duck", "Pho", "Pizza", "Pork chop", "Poutine", "Prime rib", "Pulled pork sandwich", "Ramen", "Ravioli", "Red velvet cake", "Risotto", "Samosa", "Sashimi", "Scallops", "Seaweed salad", "Shrimp and grits", "Spaghetti bolognese", "Spaghetti carbonara", "Spring rolls", "Steak", "Strawberry shortcake", "Sushi", "Tacos", "Takoyaki", "Tiramisu", "Tuna tartare", "Waffles"]\n\npet37_classes = []\ntry:\n    _p = os.path.join(root, "oxford-iiit-pet/annotations", "test.txt")\n    if os.path.exists(_p):\n        _ids, _labs = [], []\n        with open(_p) as _f:\n            for _line in _f:\n                _iid, _lab, *_ = _line.strip().split()\n                _ids.append(_iid)\n                _labs.append(int(_lab) - 1)\n        pet37_classes = [" ".join(p.title() for p in c.split("_")) for c, _ in sorted({{(i.rsplit("_", 1)[0], l) for i, l in zip(_ids, _labs)}}, key=lambda x: x[1])]\n    else:\n        warnings.warn(f"pet37 not found at {{_p}}")\nexcept Exception as e:\n    warnings.warn(f"pet37: {{e}}")\n\ndef get_classnames(set_id):\n    dataset_class_map = {{"bird200": bird200_classes, "car196": car196_classes, "food101": food101_classes, "pet37": pet37_classes, "CIFAR-100": cifar100_classes, "CIFAR-100-C": cifar100_classes, "CIFAR-100-C-OOD": cifar100_classes, "CIFAR-10": cifar10_classes, "CIFAR-10-C": cifar10_classes, "CIFAR-10-C-OOD": cifar10_classes}}\n    if set_id in fewshot_datasets:\n        classnames = eval(f"{{set_id.lower()}}_classes")\n    elif set_id in dataset_class_map:\n        classnames = dataset_class_map[set_id]\n    elif set_id in ["A", "R", "K", "V", "I", "ImageNet-C"]:\n        classnames_all = imagenet_classes\n        if set_id in ["A", "R", "V"]:\n            label_mask = eval(f"imagenet_{{set_id.lower()}}_mask")\n            if set_id == "R":\n                classnames = [classnames_all[i] for i, m in enumerate(label_mask) if m]\n            else:\n                classnames = [classnames_all[i] for i in label_mask]\n        else:\n            classnames = classnames_all\n    else:\n        raise ValueError(f"Unknown dataset ID: {{set_id}}")\n    return classnames\n'''

with open(DCN, 'w') as f:
    f.write(dcn_code)
print('Rewrote dataset_class_names.py')

for d in ['data/noisy_data_idx/0.33', 'data/noisy_data_idx/1.0', 'data/noisy_data_idx/3.0',
          'results/main_results', 'data_analysis/score', 'data_analysis/conf_pkl']:
    os.makedirs(os.path.join(REPO, d), exist_ok=True)
print('Dirs ready.')

Patched default_configs.py
Rewrote dataset_class_names.py
Dirs ready.


In [4]:
#@title **3. Write enhanced_classifier.py + patch OODDetector**

with open(os.path.join(REPO, 'clip/enhanced_classifier.py'), 'w') as f:
    f.write('import torch\nimport torch.nn as nn\nimport torch.nn.functional as F\n\nclass EnhancedOODDetector(nn.Module):\n    def __init__(self, input_size=512, hidden_size=128, use_augmented_features=True):\n        super().__init__()\n        self.use_augmented_features = use_augmented_features\n        dim = input_size + 4 if use_augmented_features else input_size\n        self.net = nn.Sequential(nn.Linear(dim, hidden_size), nn.ReLU(inplace=True), nn.Linear(hidden_size, 2))\n\n    def forward(self, x, clip_output=None):\n        if self.use_augmented_features and clip_output is not None:\n            aug = self._aug(x, clip_output)\n            x = torch.cat([x, aug], dim=-1)\n        return self.net(x)\n\n    def _aug(self, feats, clip_out):\n        norm = torch.norm(feats, dim=-1, keepdim=True)\n        p = F.softmax(clip_out, dim=-1)\n        mcm = p.max(dim=-1, keepdim=True)[0]\n        ent = -(p * F.log_softmax(clip_out, dim=-1)).sum(dim=-1, keepdim=True)\n        t2 = p.topk(min(2, p.shape[-1]), dim=-1)[0]\n        gap = (t2[:, 0] - t2[:, 1]).unsqueeze(-1) if t2.shape[-1] >= 2 else t2[:, 0:1]\n        return torch.cat([norm, mcm, ent, gap], dim=-1)\n')

clf_path = os.path.join(REPO, 'clip/classifier.py')
with open(clf_path, 'r') as f:
    clf_text = f.read()
old_block = 'class OODDetector(nn.Module):\n    def __init__(self, input_size=512, hidden_size=256):\n        super(OODDetector, self).__init__()\n        self.fc = nn.Linear(input_size, 2)\n\n    def forward(self, x):\n        x = self.fc(x)\n        return x'
new_block = 'class OODDetector(nn.Module):\n    def __init__(self, input_size=512, hidden_size=256):\n        super(OODDetector, self).__init__()\n        self.fc = nn.Linear(input_size, 2)\n\n    def forward(self, x, clip_output=None):\n        x = self.fc(x)\n        return x'
if 'def forward(self, x, clip_output=None)' not in clf_text:
    clf_text = clf_text.replace(old_block, new_block)
    with open(clf_path, 'w') as f:
        f.write(clf_text)
    print('Patched OODDetector')
else:
    print('Already patched.')

Patched OODDetector


In [5]:
#@title **4. Write MASTER zs_noisytta.py**

orig = os.path.join(REPO, 'ttda_method/zs_noisytta.py')
bak = os.path.join(REPO, 'ttda_method/zs_noisytta_ORIGINAL.py')
if not os.path.exists(bak):
    shutil.copy2(orig, bak)

MASTER = (
    'import torch\n'
    'import torch.nn.parallel\n'
    'import torch.nn.functional as F\n'
    'import torch.nn as nn\n'
    'import torch.optim\n'
    'import torch.utils.data\n'
    'import torch.utils.data.distributed\n'
    'import numpy as np\n'
    'from collections import deque\n'
    'import copy\n'
    '\n'
    'from .zs_clip import ZeroShotCLIP\n'
    'from clip.classifier import OODDetector\n'
    'from clip.enhanced_classifier import EnhancedOODDetector\n'
    'from utils.utils import *\n'
    '\n\n'
    'class ZeroShotNTTA(ZeroShotCLIP):\n'
    '    def __init__(self, *args, **kwargs):\n'
    '        super().__init__(*args, **kwargs)\n'
    '        fd = {"ViT-L/14": 768, "ViT-B/16": 512}.get(self.args.model.arch, 1024)\n'
    '        dev = self.model.image_encoder.conv1.weight.device\n'
    '        enh = getattr(self.args.inference, "use_enhanced_detector", False)\n'
    '        hs = getattr(self.args.inference, "detector_hidden_size", 128)\n'
    '        if enh:\n'
    '            self.ood_net = EnhancedOODDetector(fd, hs, True).to(dev)\n'
    '        else:\n'
    '            self.ood_net = OODDetector(fd).to(dev)\n'
    '        self.use_soft = getattr(self.args.inference, "use_soft_labels", False)\n'
    '        self.soft_alpha = getattr(self.args.inference, "soft_label_alpha", 10.0)\n'
    '        self.adaptive_alpha = getattr(self.args.inference, "adaptive_alpha", False)\n'
    '        self.asymmetric = getattr(self.args.inference, "asymmetric_soft", False)\n'
    '        self.w_id = getattr(self.args.inference, "w_id", 1.0)\n'
    '        self.w_ood = getattr(self.args.inference, "w_ood", 1.0)\n'
    '        self.training_margin = getattr(self.args.inference, "training_margin", 0.0)\n'
    '        self.use_smooth = getattr(self.args.inference, "use_smooth_transition", False)\n'
    '        self.ramp = getattr(self.args.inference, "transition_ramp_steps", 5)\n'
    '        self.ensemble_weight = getattr(self.args.inference, "ensemble_weight", 0.0)\n'
    '        self.deferral_margin = getattr(self.args.inference, "deferral_margin", 0.0)\n'
    '        self.kl_loss = nn.KLDivLoss(reduction="batchmean")\n'
    '        self.ce_loss_none = nn.CrossEntropyLoss(reduction="none")\n'
    '        self.ce_loss = nn.CrossEntropyLoss()\n'
    '        self.criterion = nn.CrossEntropyLoss()\n'
    '        self.optimizer = get_optimizer(self.args, self.ood_net.parameters(), lr=self.args.optim.lr)\n'
    '        self.os_detector_queue = []\n'
    '        self.loss_history = []\n'
    '        ql = self.args.inference.ttda_queue_length\n'
    '        self.queues = {"ood_detector_out_queue": deque(maxlen=ql), "unseen_mask_queue": deque(maxlen=ql), "target_queue": deque(maxlen=ql), "clip_output_queue": deque(maxlen=ql)}\n'
    '        self.ttda_queue = []\n'
    '\n'
    '    def _get_otsu_threshold(self, queue):\n'
    '        if len(queue) < 10: return 0.5\n'
    '        arr = np.array(queue)\n'
    '        tr = np.arange(0, 1, 0.01)\n'
    '        cs = [compute_os_variance(arr, t) for t in tr]\n'
    '        return tr[np.argmin(cs)]\n'
    '\n'
    '    def get_unseen_mask(self, clip_output, image, image_feature_raw, step, target):\n'
    '        unseen_mask = super().get_unseen_mask(clip_output, image, image_feature_raw, step, target)\n'
    '        self.model.eval()\n'
    '        self.ood_net.train()\n'
    '        with torch.enable_grad():\n'
    '            if self.args.inference.batch_size == 1:\n'
    '                ood_out = self.ood_net(image_feature_raw, clip_output=clip_output)\n'
    '                self.ttda_queue.extend(ood_out)\n'
    '                self.queues["ood_detector_out_queue"].append(ood_out)\n'
    '                self.queues["unseen_mask_queue"].append(unseen_mask)\n'
    '                self.queues["target_queue"].append(target)\n'
    '                self.queues["clip_output_queue"].append(clip_output)\n'
    '                if step != 0 and step % self.args.inference.ttda_queue_length == 0:\n'
    '                    bdo = torch.stack(list(self.queues["ood_detector_out_queue"]), 0).squeeze(1)\n'
    '                    bum = torch.stack(list(self.queues["unseen_mask_queue"]), 0).squeeze(1)\n'
    '                    btg = torch.stack(list(self.queues["target_queue"]), 0).squeeze(1)\n'
    '                    bco = torch.stack(list(self.queues["clip_output_queue"]), 0).squeeze(1)\n'
    '                    self.update_detector(bdo, bum, btg, step, bco)\n'
    '            else:\n'
    '                ood_out = self.ood_net(image_feature_raw, clip_output=clip_output)\n'
    '                self.update_detector(ood_out, unseen_mask, target, step, clip_output)\n'
    '        bs = self.args.inference.batch_size\n'
    '        ql = self.args.inference.ttda_queue_length if bs == 1 else bs\n'
    '        start = self.args.inference.using_ttda_step * ql\n'
    '        cur = step * bs\n'
    '        if cur > start:\n'
    '            pred = F.softmax(ood_out, 1)\n'
    '            det_score = pred[:, 0]\n'
    '            self.os_detector_queue.extend(det_score.detach().cpu().tolist())\n'
    '            self.os_detector_queue = self.os_detector_queue[-self.args.inference.queue_length:]\n'
    '            if self.args.inference.threshold_type == "adaptive":\n'
    '                thr = self._get_otsu_threshold(self.os_detector_queue)\n'
    '            else:\n'
    '                thr = self.args.inference.fixed_threshold\n'
    '            print(thr)\n'
    '            unseen_mask = (det_score > thr)\n'
    '            if self.use_smooth:\n'
    '                a = min(1.0, (cur - start) / max(ql * self.ramp, 1))\n'
    '                if a < 1.0:\n'
    '                    cp = F.softmax(clip_output, 1)\n'
    '                    cs = 1 - cp.max(1)[0]\n'
    '                    unseen_mask = (a * det_score + (1 - a) * cs > thr)\n'
    '            if self.ensemble_weight > 0:\n'
    '                w = self.ensemble_weight\n'
    '                cp = F.softmax(clip_output, 1)\n'
    '                unseen_mask = ((1 - w) * det_score + w * (1 - cp.max(1)[0]) > thr)\n'
    '            if self.deferral_margin > 0:\n'
    '                unc = torch.abs(pred[:, 0] - 0.5) < self.deferral_margin\n'
    '                if unc.any():\n'
    '                    cp = F.softmax(clip_output, 1)\n'
    '                    ct = self._get_otsu_threshold(self.os_inference_queue)\n'
    '                    unseen_mask[unc] = ((1 - cp.max(1)[0]) > ct)[unc]\n'
    '            return unseen_mask, pred[:, 1]\n'
    '        return unseen_mask\n'
    '\n'
    '    def update_detector(self, ood_out, unseen_mask, target, step, clip_output=None):\n'
    '        logit = F.softmax(clip_output, dim=1)\n'
    '        conf, _ = logit.max(1)\n'
    '        unseen_mask[target == -1000] = True\n'
    '        if self.training_margin > 0:\n'
    '            mcm = logit.max(1)[0]\n'
    '            mcm_thr = 1.0 - self._get_otsu_threshold(self.os_inference_queue) if len(self.os_inference_queue) > 10 else 0.5\n'
    '            confident = torch.abs(mcm - mcm_thr) > self.training_margin\n'
    '            confident[target == -1000] = True\n'
    '            si = torch.where(~unseen_mask & confident)[0]\n'
    '            so = torch.where(unseen_mask & confident)[0]\n'
    '        else:\n'
    '            si = torch.where(~unseen_mask)[0]\n'
    '            so = torch.where(unseen_mask)[0]\n'
    '        sa = torch.cat([si, so])\n'
    '        if len(sa) < 4: return\n'
    '        if self.use_soft and clip_output is not None:\n'
    '            loss = self._soft_loss(ood_out, target, clip_output, si, so, sa)\n'
    '        elif self.w_id != 1.0 or self.w_ood != 1.0:\n'
    '            lab = torch.cat([torch.ones(len(si)), torch.zeros(len(so))]).cuda()\n'
    '            ps = self.ce_loss_none(ood_out[sa], lab.long())\n'
    '            wt = torch.ones_like(ps)\n'
    '            wt[:len(si)] = self.w_id\n'
    '            wt[len(si):] = self.w_ood\n'
    '            loss = (ps * wt).mean()\n'
    '        else:\n'
    '            lab = torch.cat([torch.ones(len(si)), torch.zeros(len(so))]).cuda()\n'
    '            loss = self.ce_loss(ood_out[sa], lab.long())\n'
    '        self.optimizer.zero_grad()\n'
    '        loss.backward()\n'
    '        self.loss_history.append(loss.item())\n'
    '        self.optimizer.step()\n'
    '\n'
    '    def _soft_loss(self, ood_out, target, clip_output, si, so, sa):\n'
    '        if self.adaptive_alpha:\n'
    '            if len(self.os_inference_queue) > 10:\n'
    '                arr = np.array(self.os_inference_queue)\n'
    '                tr = np.arange(0, 1, 0.01)\n'
    '                cs = [compute_os_variance(arr, t) for t in tr]\n'
    '                a = 20.0 - 17.0 * min(cs[np.argmin(cs)] / 0.083, 1.0)\n'
    '            else: a = 10.0\n'
    '        else: a = self.soft_alpha\n'
    '        p = F.softmax(clip_output, 1)\n'
    '        mcm = p.max(1)[0]\n'
    '        mcm_thr = 1.0 - self._get_otsu_threshold(self.os_inference_queue)\n'
    '        soft = torch.zeros(len(sa), 2, device=ood_out.device)\n'
    '        if len(si) > 0:\n'
    '            c = torch.sigmoid(a * (mcm[si] - mcm_thr))\n'
    '            soft[:len(si), 1] = c\n'
    '            soft[:len(si), 0] = 1 - c\n'
    '        if len(so) > 0:\n'
    '            off = len(si)\n'
    '            if self.asymmetric:\n'
    '                soft[off:, 0] = 1.0\n'
    '                soft[off:, 1] = 0.0\n'
    '            else:\n'
    '                c = torch.sigmoid(a * (mcm_thr - mcm[so]))\n'
    '                c[target[so] == -1000] = 1.0\n'
    '                soft[off:, 0] = c\n'
    '                soft[off:, 1] = 1 - c\n'
    '        soft = soft / soft.sum(1, keepdim=True).clamp(min=1e-8)\n'
    '        lp = F.log_softmax(ood_out[sa], 1)\n'
    '        return self.kl_loss(lp, soft.detach())\n'
)

with open(orig, 'w') as f:
    f.write(MASTER)
print('Wrote MASTER zs_noisytta.py')

Wrote MASTER zs_noisytta.py


In [6]:
#@title **5. Setup Places dataset**
try:
    !pip install -q pytorch-ood
    import pytorch_ood.dataset.img as ood_data
    ds = ood_data.Places365(root=os.path.join(DATA, '_ood_raw'), download=True)
    !mkdir -p {DATA}/ImageNet_OOD_dataset
    !ln -sf {DATA}/_ood_raw/places365 {DATA}/ImageNet_OOD_dataset/Places
    print('Places ready.')
except Exception as e:
    print(f'Places setup failed: {e}')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.7/42.7 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.2/183.2 kB 16.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 45.7 MB/s eta 0:00:00
Places setup failed: [Errno 2] No such file or directory: '/content/datasets/_ood_raw'


In [7]:
#@title **6. Write all bs=1 configs for CIFAR-100+Places**

def cfg1(name, enhanced=False, soft=False, adapt_a=False, asym=False,
         smooth=False, w_id=1.0, margin=0.0, ens=0.0, defer=0.0):
    with open(os.path.join(REPO, f'configs/{name}'), 'w') as f:
        f.write(f'''from configs.default_configs import get_default_configs\ndef get_config():\n    c = get_default_configs()\n    c.method = "ZS-NTTA"\n    c.training.save_ckpt = True\n    c.training.gaussian_sampling = False\n    i = c.inference\n    i.ttda_queue_length = 128\n    i.top = 1.\n    i.update_classifier = "None"\n    i.inject_noise_type = "gaussian"\n    i.using_ttda_step = 10\n    i.gaussian_rate = 0.125\n    i.batch_size = 1\n    i.use_enhanced_detector = {enhanced}\n    i.detector_hidden_size = 128\n    i.use_soft_labels = {soft}\n    i.soft_label_alpha = 10.0\n    i.adaptive_alpha = {adapt_a}\n    i.asymmetric_soft = {asym}\n    i.use_smooth_transition = {smooth}\n    i.transition_ramp_steps = 5\n    i.w_id = {w_id}\n    i.w_ood = 1.0\n    i.training_margin = {margin}\n    i.ensemble_weight = {ens}\n    i.deferral_margin = {defer}\n    o = c.optim\n    o.classifier_lr = 0.0005\n    o.lr = 0.0005\n    o.weight_decay = 0\n    o.optimizer = "Adam"\n    o.beta1 = 0.9\n    return c\n''')

cfg1('bs1_base.py')
cfg1('bs1_det.py',        enhanced=True)
cfg1('bs1_soft.py',       soft=True)
cfg1('bs1_smooth.py',     smooth=True)
cfg1('bs1_v2.py',         enhanced=True, soft=True, smooth=True)
cfg1('bs1_det_soft.py',   enhanced=True, soft=True)
cfg1('bs1_asym_soft.py',  soft=True, asym=True)
cfg1('bs1_wid2.py',       w_id=2.0)
cfg1('bs1_wid3.py',       w_id=3.0)
cfg1('bs1_ens.py',        ens=0.3)
cfg1('bs1_defer.py',      defer=0.15)
cfg1('bs1_3way005.py',    margin=0.05)

print('All 12 bs=1 configs created.')

All 12 bs=1 configs created.


In [8]:
#@title **7. Experiment runner**

def run(config, id_set, ood_set, gpu=0, label=None):
    label = label or config.replace('.py', '')
    print(f'\n{"="*55}\n  {label}  |  ID={id_set}  OOD={ood_set}\n{"="*55}')
    rd = os.path.join(REPO, f'results/main_results/{id_set}')
    if os.path.isdir(rd):
        for sub in os.listdir(rd):
            sd = os.path.join(rd, sub)
            if os.path.isdir(sd):
                for ff in os.listdir(sd):
                    if ood_set in ff:
                        os.remove(os.path.join(sd, ff))
    cmd = ['python', 'main.py', f'--config=configs/{config}',
           f'--test_set={id_set}', f'--OOD_set={ood_set}', f'--gpu={gpu}']
    t0 = time.time()
    try:
        r = subprocess.run(cmd, capture_output=True, text=True, timeout=7200, cwd=REPO)
        out = r.stdout + '\n' + r.stderr
    except Exception as e:
        print(f'  ERROR: {e}')
        return {'label': label, 'id': id_set, 'ood': ood_set, 'error': str(e)}
    dt = time.time() - t0
    m = {}
    for line in out.split('\n'):
        if 'ACC_S:' in line and 'ACC_N:' in line and 'ACC_H:' in line:
            ms = re.search(r'ACC_S:\s*([\d.]+)', line)
            mn = re.search(r'ACC_N:\s*([\d.]+)', line)
            mh = re.search(r'ACC_H:\s*([\d.]+)', line)
            if ms and mn and mh:
                m = {'AccS': float(ms.group(1)), 'AccN': float(mn.group(1)), 'AccH': float(mh.group(1))}
    if m:
        print(f'  AccS={m["AccS"]:.2f}  AccN={m["AccN"]:.2f}  AccH={m["AccH"]:.2f}  ({dt:.0f}s)')
    else:
        print('  Parse failed. Last lines:')
        for l in out.strip().split('\n')[-10:]:
            if l.strip(): print(f'    {l.strip()}')
    return {'label': label, 'id': id_set, 'ood': ood_set, **m}

print('Runner ready. Timeout set to 2h for bs=1 experiments.')

Runner ready. Timeout set to 2h for bs=1 experiments.


In [9]:
#@title **8. Run ZS-CLIP first**
run('zs_clip_configs.py', 'CIFAR-100', 'Places', label='ZS-CLIP')


  ZS-CLIP  |  ID=CIFAR-100  OOD=Places
  Parse failed. Last lines:
    File "/usr/local/lib/python3.12/dist-packages/torchvision/datasets/folder.py", line 149, in __init__
    classes, class_to_idx = self.find_classes(self.root)
    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    File "/usr/local/lib/python3.12/dist-packages/torchvision/datasets/folder.py", line 234, in find_classes
    return find_classes(directory)
    ^^^^^^^^^^^^^^^^^^^^^^^
    File "/usr/local/lib/python3.12/dist-packages/torchvision/datasets/folder.py", line 41, in find_classes
    classes = sorted(entry.name for entry in os.scandir(directory) if entry.is_dir())
    ^^^^^^^^^^^^^^^^^^^^^
    FileNotFoundError: [Errno 2] No such file or directory: '/content/datasets/ImageNet_OOD_dataset/Places'


{'label': 'ZS-CLIP', 'id': 'CIFAR-100', 'ood': 'Places'}

In [ ]:
#@title **9. Run ALL methods on CIFAR-100+Places (bs=1)**

CFGS = [
    ('bs1_base.py',       'Base'),
    ('bs1_det.py',        'Enh MLP'),
    ('bs1_soft.py',       'Sym Soft'),
    ('bs1_smooth.py',     'Smooth'),
    ('bs1_v2.py',         'v2 (all)'),
    ('bs1_det_soft.py',   'Det+Soft'),
    ('bs1_asym_soft.py',  'Asym Soft'),
    ('bs1_wid2.py',       'w_id=2'),
    ('bs1_wid3.py',       'w_id=3'),
    ('bs1_ens.py',        'Ensemble'),
    ('bs1_defer.py',      'Deferral'),
    ('bs1_3way005.py',    '3way=0.05'),
]

R = []
for i, (c, lbl) in enumerate(CFGS):
    print(f'\n--- {i+1}/{len(CFGS)} ---')
    r = run(c, 'CIFAR-100', 'Places', label=lbl)
    R.append(r)

In [ ]:
#@title **10. Results table**

print(f'\n{"="*60}')
print(f'  CIFAR-100 + Places (bs=1, paper-matching settings)')
print(f'{"="*60}')
print(f'{"Method":<14s} {"AccS":>8s} {"AccN":>8s} {"AccH":>8s}  {"Delta":>7s}')
print(f'{"-"*60}')

base_h = None
for r in R:
    if r['label'] == 'Base' and 'AccH' in r:
        base_h = r['AccH']

for r in R:
    if 'AccH' in r:
        delta = r['AccH'] - base_h if base_h else 0
        sign = '+' if delta >= 0 else ''
        print(f'{r["label"]:<14s} {r["AccS"]:>8.2f} {r["AccN"]:>8.2f} {r["AccH"]:>8.2f}  {sign}{delta:>6.2f}')
    else:
        print(f'{r["label"]:<14s}   FAILED')